In [1]:
import json
import datetime
from dateutil import parser
import os

In [ ]:
with open("A-mem/results_robust_gpt-4o-mini.json", "r") as file:
    example = json.load(file)

In [4]:
example['aggregate_metrics'].keys()

dict_keys(['overall', 'category_1', 'category_2', 'category_3', 'category_4', 'category_5'])

In [5]:
metrics = list(example['aggregate_metrics']['overall'].keys())

In [6]:
remove = [
    'rouge1_f',
    'bleu2',
    'bleu3',
    'bleu4',
    'bert_precision',
    'bert_recall'
]

for metric in remove:
    metrics.remove(metric)

In [7]:
metrics[5:7] = metrics[5:7][::-1]

In [8]:
METRICS = {}

for metric in metrics:
    if 'rouge' in metric:
        metric_name = metric.split('_')[0]
    else:
        metric_name = metric
    if "rouge" in metric or "bleu" in metric:
        metric_name = '-'.join([metric_name[:-1].upper(), metric_name[-1].upper()])
    elif '_' in metric_name:
        metric_name = metric.split('_')
        sep = '-' if 'f1' in metric else " "
        if 'bert' in metric:
            metric_name = sep.join([metric_name[0].upper(), metric_name[-1].capitalize()])
        else:
            metric_name = sep.join([metric_name[0].capitalize(), metric_name[-1].capitalize()])
    else:
        metric_name = metric_name.upper()
    METRICS[metric] = metric_name

METRICS

{'exact_match': 'Exact Match',
 'f1': 'F1',
 'rouge2_f': 'ROUGE-2',
 'rougeL_f': 'ROUGE-L',
 'bleu1': 'BLEU-1',
 'meteor': 'METEOR',
 'bert_f1': 'BERT-F1',
 'sbert_similarity': 'SBERT Similarity'}

In [9]:
categories = [f'category_{i+1}' for i in range(5)] + ['overall']
category_names = ["Multi Hop", "Temporal", "Open Domain", "Single Hop", "Adversial", "Average"]
CATEGORIES = {
    category: name
    for name, category in zip(category_names, categories)
}

In [11]:
def format_best(rep_score, base_score):
    if rep_score * 100 > base_score:
        rep_score = f"\\textbf{{{rep_score * 100:.2f}}}"
    else:
        base_score = f"\\textbf{{{base_score:.2f}}}"
    return rep_score, base_score

In [12]:
def format_table(tabular):
    table = f"""\\begin{{table}}[]
    \\centering
    \\resizebox{{\\linewidth}}{{!}}{{
        \\begin{{tabular}}{{l|cc|cccc|cc}}
        {"".join(tabular)}
        \\end{{tabular}}
}}
\\end{{table}}
    """.strip()
    
    print(table)

In [37]:
def write_table(results, category):
    tabular = ["\\toprule"]
    header = "\\textbf{Model}\t"
    for metric in METRICS.values():
        header += f"\t&\t\\textbf{{{metric}}}"
    header += "\t\\\\"
    category_name = CATEGORIES[category]
    tabular.append(f"\\multicolumn{{9}}{{c}}{{\\textbf{{{category_name}}}}}\t\\\\")
    tabular.append('\\midrule')
    tabular.append(header)
    models = set()
    if category == 'overall':
        tabular.append('\\midrule')
    for result in results:
        model = result['model'].split('/')[-1]
        mode = 'A-mem' if result.get('aggregate_metrics') is not None else None
        if not model in models:
            models.add(model)
            index = None
        else:
            for i, _row in enumerate(tabular):
                if model in _row:
                    index = i
            model = "\\quad + A-mem"
        row = f"{model}\t"
        scores = result.get('aggregate_metrics') or result.get('aggregated_scores')
        scores = scores[category]
        for metric in METRICS.keys():
            score = scores[metric]['mean']
            row += f"&\t{score * 100:.2f}\t"
        row += '\\\\'
        if index is not None:
            tabular.insert(index + 1, row)
        else:
            tabular.append(row)
    tabular.append('\\bottomrule')

    return (
        '\t' + row + '\n' if not i == 0 else row + '\n' for i, row in enumerate(tabular)
    )

In [38]:
results_dir = "results/"
results_files = [file for file in os.listdir(results_dir) if not "thinking" in file]
results = {}
for result_file in results_files:
    result_path = results_dir + result_file
    with open(result_path, "r") as file:
        results[result_file] = json.load(file)

In [39]:
for category in categories:
    format_table(write_table(list(results.values()), category))
    print('\n\n')

\begin{table}[]
    \centering
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|cc|cccc|cc}
        \toprule
	\multicolumn{9}{c}{\textbf{Multi Hop}}	\\
	\midrule
	\textbf{Model}		&	\textbf{Exact Match}	&	\textbf{F1}	&	\textbf{ROUGE-2}	&	\textbf{ROUGE-L}	&	\textbf{BLEU-1}	&	\textbf{METEOR}	&	\textbf{BERT-F1}	&	\textbf{SBERT Similarity}	\\
	Qwen3-4B-Instruct-2507	&	7.45	&	32.63	&	13.48	&	29.02	&	25.26	&	23.39	&	88.11	&	58.13	\\
	\quad + A-mem	&	0.00	&	7.76	&	1.09	&	7.71	&	6.72	&	4.54	&	85.42	&	25.61	\\
	Qwen3-30B-A3B-Instruct-2507	&	10.28	&	43.97	&	17.53	&	40.93	&	34.06	&	25.76	&	89.82	&	66.79	\\
	\quad + A-mem	&	0.00	&	10.60	&	1.91	&	10.34	&	8.57	&	6.29	&	86.83	&	30.21	\\
	\bottomrule

        \end{tabular}
}
\end{table}



\begin{table}[]
    \centering
    \resizebox{\linewidth}{!}{
        \begin{tabular}{l|cc|cccc|cc}
        \toprule
	\multicolumn{9}{c}{\textbf{Temporal}}	\\
	\midrule
	\textbf{Model}		&	\textbf{Exact Match}	&	\textbf{F1}	&	\textbf{ROUGE-2}	&	\textbf{ROUGE-L

In [20]:
keys = ["model", "dataset", "total_questions", "individual_results"]

In [17]:
result_dir = "individual_results"
os.makedirs(result_dir, exist_ok=True)
for result_file, result in results.items():
    ind_result = {
        key: result[key]
        for key in keys
    }
    with open(f"{result_dir}/{result_file}", "w") as file:
        json.dump(ind_result, file, indent=2)